#section 1 : Data Cleaning and preparation

In [ ]:

import numpy as np
import pandas as pd

In [3]:
#load data
df= pd.read_csv("C:\\Users\\shrir\\Downloads\\Loan_default.csv")

In [4]:
#M/D/YYY to DD/MM/YYYY
df['LoanDate'] = pd.to_datetime(df['Loan Date (DD/MM/YYYY)'], infer_datetime_format = True)
df['LoanYear'] = df['LoanDate'].dt.year
df['LoanMonth'] = df['LoanDate'].dt.month

C:\Users\shrir\AppData\Local\Temp\ipykernel_14348\2460098993.py:2: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df['LoanDate'] = pd.to_datetime(df['Loan Date (DD/MM/YYYY)'], infer_datetime_format = True)


In [5]:
df['LoanDate'] = pd.to_datetime(df['Loan Date (DD/MM/YYYY)'], dayfirst=True)

df['LoanYear'] = df['LoanDate'].dt.year
df['LoanMonth'] = df['LoanDate'].dt.month

C:\Users\shrir\AppData\Local\Temp\ipykernel_14348\2025347971.py:1: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['LoanDate'] = pd.to_datetime(df['Loan Date (DD/MM/YYYY)'], dayfirst=True)


In [6]:
#income bands
df['IncomeBand'] = pd.cut(
    df['Income'],
    bins=[0, 30000, 60000, 90000, 120000, 160000],
    labels=['<30K', '30–60K', '60–90K', '90–120K', '>120K']
)

In [7]:
#credit score bands
df['CreditBand'] = pd.cut(
    df['CreditScore'],
    bins=[299, 579, 669, 739, 799, 850],
    labels=['Poor (300–579)', 'Fair (580–669)', 'Good (670–739)',
            'Very Good (740–799)', 'Exceptional (800–850)']
)

In [8]:
# DTI Bands
df['DTIBand'] = pd.cut(
    df['DTIRatio'],
    bins=[0, 0.20, 0.35, 0.50, 0.65, 1.0],
    labels=['Very Low (<20%)', 'Low (20–35%)', 'Medium (35–50%)',
            'High (50–65%)', 'Very High (>65%)']
)

In [9]:
#Interest Rate Bands
df['RateBand'] = pd.cut(
    df['InterestRate'],
    bins=[0, 7, 13, 18, 25.1],
    labels=['Low (≤7%)', 'Medium (7–13%)', 'High (13–18%)', 'Very High (>18%)']
)

In [10]:
#Employement risk category
emp_risk = {
    'Full-time': 'Stable',
    'Part-time': 'Moderate',
    'Self-employed': 'Variable',
    'Unemployed': 'High Risk'
}
df['EmpRiskCategory'] = df['EmploymentType'].map(emp_risk)


In [11]:
#Loan size category
df['LoanSizeCategory'] = pd.cut(
    df['LoanAmount'],
    bins=[0, 50000, 100000, 150000, 200000, 250001],
    labels=['Micro (<50K)', 'Small (50–100K)', 'Medium (100–150K)',
            'Large (150–200K)', 'Very Large (>200K)']
)

In [12]:
#outliers
for col in ['Income', 'LoanAmount', 'InterestRate', 'DTIRatio']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 3*IQR, Q3 + 3*IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {n_out} outliers beyond 3×IQR")

Income: 0 outliers beyond 3×IQR
LoanAmount: 0 outliers beyond 3×IQR
InterestRate: 0 outliers beyond 3×IQR
DTIRatio: 0 outliers beyond 3×IQR


In [13]:
#saving the dataset
df.to_csv("loan_clean.csv", index=False)
print(f"Clean dataset saved: {df.shape}")

Clean dataset saved: (255347, 28)


#Section 2 : Risk & Business analytics

In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [15]:
df = pd.read_csv("loan_clean.csv")


In [16]:
#overall default rate
overall_dr = df['Default'].mean()
print(f"Portfolio default rate: {overall_dr:.2%}")

Portfolio default rate: 11.61%


In [17]:
#DEFAULT RATE BY SEGMENT
def default_profile(df, col):
    return (df.groupby(col, observed=True)['Default']
              .agg(Count='count', Defaults='sum', DefaultRate='mean')
              .sort_values('DefaultRate', ascending=False)
              .assign(DefaultRate=lambda x: x['DefaultRate'].round(4)))

print("\n── Default Rate by Income Band ──")
print(default_profile(df, 'IncomeBand'))

print("\n── Default Rate by Employment Type ──")
print(default_profile(df, 'EmploymentType'))

print("\n── Default Rate by Loan Purpose ──")
print(default_profile(df, 'LoanPurpose'))

print("\n── Default Rate by Credit Band ──")
print(default_profile(df, 'CreditBand'))

print("\n── Default Rate by DTI Band ──")
print(default_profile(df, 'DTIBand'))

print("\n── Default Rate by Interest Rate Band ──")
print(default_profile(df, 'RateBand'))


── Default Rate by Income Band ──
            Count  Defaults  DefaultRate
IncomeBand                              
<30K        28405      6237       0.2196
30–60K      56650      7241       0.1278
60–90K      56873      5779       0.1016
90–120K     56776      5281       0.0930
>120K       56643      5115       0.0903

── Default Rate by Employment Type ──
                Count  Defaults  DefaultRate
EmploymentType                              
Unemployed      63824      8650       0.1355
Part-time       64161      7677       0.1197
Self-employed   63706      7302       0.1146
Full-time       63656      6024       0.0946

── Default Rate by Loan Purpose ──
             Count  Defaults  DefaultRate
LoanPurpose                              
Business     51298      6323       0.1233
Auto         50844      6041       0.1188
Education    51005      6038       0.1184
Other        50914      6002       0.1179
Home         51286      5249       0.1023

── Default Rate by Credit Band ──
    

In [18]:
# cohort analysis
cohort = (df.groupby('LoanYear')['Default']
            .agg(Loans='count', Defaults='sum', DefaultRate='mean')
            .reset_index())
print("\n── Default Rate by Loan Year ──")
print(cohort)


── Default Rate by Loan Year ──
   LoanYear  Loans  Defaults  DefaultRate
0      2013  42785      4973     0.116232
1      2014  42122      4845     0.115023
2      2015  42521      4976     0.117025
3      2016  42705      5017     0.117480
4      2017  42377      4875     0.115039
5      2018  42837      4967     0.115951


In [19]:
# multi_dimensional segment analysis
segment_cross = (df.groupby(['EmploymentType', 'DTIBand'], observed=True)['Default']
                   .mean()
                   .unstack()
                   .round(4))
print("\n── Default Rate: Employment × DTI ──")
print(segment_cross)


── Default Rate: Employment × DTI ──
DTIBand         High (50–65%)  Low (20–35%)  Medium (35–50%)  \
EmploymentType                                                 
Full-time              0.0944        0.0937           0.0900   
Part-time              0.1252        0.1119           0.1193   
Self-employed          0.1183        0.1092           0.1162   
Unemployed             0.1372        0.1318           0.1353   

DTIBand         Very High (>65%)  Very Low (<20%)  
EmploymentType                                     
Full-time                 0.0994           0.0918  
Part-time                 0.1264           0.1074  
Self-employed             0.1222           0.0973  
Unemployed                0.1445           0.1180  


In [20]:
#correlation analysis
numeric_cols = ['Age','Income','LoanAmount','CreditScore',
                'MonthsEmployed','InterestRate','DTIRatio','Default']
corr = df[numeric_cols].corr()['Default'].drop('Default').sort_values()
print("\n── Correlation with Default ──")
print(corr.round(4))


── Correlation with Default ──
Age              -0.1678
Income           -0.0991
MonthsEmployed   -0.0974
CreditScore      -0.0342
DTIRatio          0.0192
LoanAmount        0.0867
InterestRate      0.1313
Name: Default, dtype: float64


#Rule based risk segmentation

In [21]:
def assign_risk_score(row):
    """
    Rule-based scoring: max possible = 10 points.
    Lower score = lower risk. Threshold: Low ≤3, Medium 4–6, High ≥7.
    """
    score = 0

    # Credit Score (0–4 points) — most important variable
    if row['CreditScore'] < 580:    score += 4   # Poor
    elif row['CreditScore'] < 670:  score += 3   # Fair
    elif row['CreditScore'] < 740:  score += 2   # Good
    elif row['CreditScore'] < 800:  score += 1   # Very Good
    # else: Exceptional = 0 points

    # DTI Ratio (0–3 points)
    if row['DTIRatio'] > 0.65:      score += 3   # Very High
    elif row['DTIRatio'] > 0.50:    score += 2   # High
    elif row['DTIRatio'] > 0.35:    score += 1   # Medium
    # else: Low/Very Low = 0

    # Employment Type (0–2 points)
    if row['EmploymentType'] == 'Unemployed':     score += 2
    elif row['EmploymentType'] == 'Self-employed': score += 1
    # Part-time = 0.5 could also be applied; we round to 0 here

    # Interest Rate (0–1 point) — acts as a tiebreaker
    if row['InterestRate'] > 18:    score += 1

    return score

# Apply scoring
df['RiskScore'] = df.apply(assign_risk_score, axis=1)

# Map to risk tier
def assign_tier(score):
    if score <= 3:   return 'Low Risk'
    elif score <= 6: return 'Medium Risk'
    else:            return 'High Risk'

df['RiskTier'] = df['RiskScore'].apply(assign_tier)

# Validate: the tiers should show VERY different default rates.
# If they don't, revise the thresholds.
validation = (df.groupby('RiskTier')['Default']
                .agg(Count='count', DefaultRate='mean')
                .sort_values('DefaultRate'))
print(validation)
# Expected: Low Risk ~5–7%, Medium ~10–15%, High ~20–30%

df.to_csv("loan_clean.csv", index=False)

              Count  DefaultRate
RiskTier                        
Low Risk      44209     0.086747
Medium Risk  128387     0.109785
High Risk     82751     0.141666
